In [4]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,AutoModelForCausalLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


In [2]:
dpo_train_data = [
    {
        "prompt": "Review: This movie is a masterpiece. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: I hated this film. Total waste of time. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    },
    {
        "prompt": "Review: Outstanding acting and brilliant plot. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Terrible acting and boring screenplay. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"

    }
]

dpo_test_data = [
    {
        "prompt": "Review: The movie was mediocre, but the ending was spectacular! Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    }
]

train_dataset = Dataset.from_list(dpo_train_data)
test_dataset = Dataset.from_list(dpo_test_data)
print("DPO dataset initialized!")

DPO dataset initialized!


In [6]:
model = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

# gpt-2 패딩 설정 조정
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
model.to(device)

# LoRA 설정
peft_config= LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r = 8,          # 저차원 랭크의 크기
    lora_alpha=32,  # 가중치 스케일 상수
    lora_dropout=0.1,
    target_modules=['c_attn'],
)

# PEFT 모델로 변환
model = get_peft_model(model, peft_config=peft_config)
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7024.04it/s]
c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


(124734720, 294912, 0.23643136409814366)

In [8]:
train_args = DPOConfig(
    output_dir = './gpt_dpo_results',
    eval_strategy='epoch',
    learning_rate=5e-4,
    num_train_epochs=2,
    beta = 0.1,
    use_cpu = True
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,  # 메모리 보존
    args = train_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'DPO training  completed : {training_time:.2f} seconds') 

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,2.737717,116.000000,-79.035309,-79.035309,0.000000,-0.004725,-0.004725,0.000000,0.000000,-7.261604,-7.261604
2,No log,0.693147,2.845660,232.000000,-78.922211,-78.922211,0.000000,-0.008807,-0.008807,0.000000,0.000000,-7.302425,-7.302425


DPO training  completed : 291.45 seconds


In [9]:
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # 생성된 텍스트 중 프롬프트 부분을 제외한 새로 추가된 토큰만 해독
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()
    return pred_text

print("\n--- Testing DPO Aligned Model ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Testing DPO Aligned Model ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: i'm not sure if it's because of

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: i'm not a fan of the movie,



In [16]:
sft_train_list = [
    {'text':f"{item['prompt']}{item['chosen']}{tokenizer.eos_token}"}
for item in dpo_train_data
]
sft_dataset = Dataset.from_list(sft_train_list)
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir="./gpt2_sft_results",
    eval_strategy="no",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    num_train_epochs=12,      # 데이터가 극소량이므로 형식을 인지하도록 에폭을 늘립니다.
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer
)

sft_start_time = time.time()
sft_trainer.train()
sft_time = time.time() - sft_start_time
print(f"SFT completed in {sft_time:.2f} seconds.")

Tokenizing train dataset: 100%|██████████| 4/4 [00:00<00:00, 263.33 examples/s]


Step,Training Loss
10,3.615102
20,3.270738


SFT completed in 555.57 seconds.


In [17]:
training_args = DPOConfig(
    output_dir="./gpt2_dpo_results",
    eval_strategy="epoch",
    learning_rate=2e-4,       # SFT 모델을 기반으로 하므로 적절한 학습률 적용
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    beta=0.1,                 # DPO margin
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,           # PEFT 활성 상태이므로 None으로 주면 베이스 모델(SFT 미수행)을 ref로 참조
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

dpo_start_time = time.time()
trainer.train()
dpo_time = time.time() - dpo_start_time
print(f"DPO completed in {dpo_time:.2f} seconds.")

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,3.913096,116.000000,-78.711121,-78.711121,0.000000,-0.001316,-0.001316,0.000000,0.000000,-6.008430,-6.008430
2,No log,0.693147,3.939592,232.000000,-79.049881,-79.049881,0.000000,0.031436,0.031436,0.000000,0.000000,-5.680908,-5.680908
3,No log,0.693147,4.012455,348.000000,-78.916336,-78.916336,0.000000,0.023969,0.023969,0.000000,0.000000,-5.755581,-5.755581


DPO completed in 443.89 seconds.


In [18]:
print("\n--- Inference after DPO (Final Alignment Check) ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Inference after DPO (Final Alignment Check) ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: very positive.
posted by: anonymous at

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: "i'm not sure what to say about

